# Solutions - Null Handling (Easy 13)


In [ ]:
from pyspark.sql import functions as F, types as T

customers = spark.table("samples.bakehouse.sales_customers")
franchises = spark.table("samples.bakehouse.sales_franchises")
tpch_customer = spark.table("samples.tpch.customer")


## Problem 1 - Count Nulls Per Column

For each column in the customers table, count nulls and calculate the null rate percentage.


In [ ]:
total = customers.count()
rows = []
for c in customers.columns:
    nc = customers.filter(F.col(c).isNull()).count()
    rows.append((c, nc, round(nc / total * 100, 2)))
result_1 = spark.createDataFrame(rows, ["column_name", "null_count", "null_rate_pct"])
result_1.display()


## Problem 2 - Filter Nulls Summary

Count customers with null vs non-null state values.


In [ ]:
null_count = customers.filter(F.col("state").isNull()).count()
not_null_count = customers.filter(F.col("state").isNotNull()).count()
result_2 = spark.createDataFrame(
    [("state_is_null", null_count), ("state_is_not_null", not_null_count)],
    ["filter_type", "customer_count"]
)
result_2.display()


## Problem 3 - Fill Nulls

Fill null values in `state` with "Unknown" and `gender` with "Not Specified".


In [ ]:
result_3 = customers.fillna({"state": "Unknown", "gender": "Not Specified"})
result_3.display()


## Problem 4 - Coalesce Fallback

Create a `location_label` using coalesce to pick the first non-null value from district, city, or a default.


In [ ]:
result_4 = franchises.select(
    "franchiseID", "name", "district", "city",
    F.coalesce(F.col("district"), F.col("city"), F.lit("Unknown")).alias("location_label")
)
result_4.display()


## Problem 5 - Drop NA Summary

Compare row counts before and after dropping rows with nulls in key columns.


In [ ]:
before = tpch_customer.count()
after = tpch_customer.dropna(subset=["c_name", "c_mktsegment"]).count()
result_5 = spark.createDataFrame(
    [("before_dropna", before), ("after_dropna", after)],
    ["step", "row_count"]
)
result_5.display()
